# Product Experimentation Framework
**Author:** Okello David  
**Role:** Backend & ML Engineer @ E&M Technology House Uganda

---

## What this notebook covers
A complete walkthrough of the A/B experimentation framework:
1. Experiment design — hypothesis, metrics, sample size calculation
2. Deterministic user assignment (control vs. treatment)
3. Simulating experiment data collection
4. Statistical significance testing (z-test and t-test)
5. Interpreting results and making a ship/no-ship decision
6. Full end-to-end example: payment confirmation screen test

> **Context:** This framework replaced ad-hoc decision-making at E&M Technology House Uganda. Before it existed, product changes shipped based on intuition. After, every significant feature change went through a structured test.

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import hashlib
import json
from scipy import stats
from dataclasses import dataclass, field
from typing import Optional
from datetime import datetime, timedelta

np.random.seed(42)
print("Setup complete")

## 1. Experiment definition

Every experiment starts with a structured hypothesis. Forcing this structure
prevents vague experiments like 'let's test the new design' that produce uninterpretable results.

In [ ]:
@dataclass
class Experiment:
    """
    Represents a single A/B experiment.
    
    Fields:
        name:             Unique identifier — used in user assignment hashing
        hypothesis:       Written hypothesis in IF/THEN/BECAUSE format
        metric:           What we're measuring (e.g. 'payment_completion_rate')
        metric_type:      'binary' (converted yes/no) or 'continuous' (amount, time)
        treatment_split:  Fraction of users in treatment group (0.5 = 50/50 split)
        min_detectable_effect: Smallest lift worth detecting (e.g. 0.05 = 5%)
        baseline_rate:    Current metric value before experiment
        start_date:       When the experiment started
        end_date:         When data collection stopped
    """
    name:                  str
    hypothesis:            str
    metric:                str
    metric_type:           str        # 'binary' or 'continuous'
    treatment_split:       float = 0.5
    min_detectable_effect: float = 0.05
    baseline_rate:         float = 0.0
    start_date:            str   = ""
    end_date:              str   = ""

    def assign_user(self, user_id: str) -> str:
        """
        Deterministically assign a user to control or treatment.
        
        WHY DETERMINISTIC?
        If we used random assignment per session, the same user might see
        'control' on Monday and 'treatment' on Tuesday. This cross-contamination
        corrupts our data — we can't compare groups cleanly.
        
        By hashing user_id + experiment_name, the same user always maps
        to the same group, regardless of when or how often they visit.
        
        Args:
            user_id: Unique user identifier
        
        Returns:
            'control' or 'treatment'
        """
        # MD5 hash of user_id + experiment name → consistent bucket
        hash_input = f"{user_id}:{self.name}".encode('utf-8')
        hash_int   = int(hashlib.md5(hash_input).hexdigest(), 16)
        bucket     = (hash_int % 100) / 100.0  # Value between 0.0 and 1.0
        
        return 'treatment' if bucket < self.treatment_split else 'control'

    def required_sample_size(self) -> dict:
        """
        Calculate minimum sample size per group before the experiment starts.
        
        WHY CALCULATE THIS UPFRONT?
        Peeking at results and stopping early when p < 0.05 inflates false positive
        rates dramatically. Committing to a sample size before starting prevents this.
        
        Uses the standard formula for two-proportion z-test power analysis.
        
        Returns:
            dict with required n per group and total n
        """
        alpha = 0.05  # Significance level (5% false positive rate)
        power = 0.80  # Statistical power (80% chance of detecting true effect)
        
        z_alpha = stats.norm.ppf(1 - alpha / 2)  # ~1.96 for 95% CI
        z_beta  = stats.norm.ppf(power)           # ~0.84 for 80% power
        
        p1 = self.baseline_rate
        p2 = self.baseline_rate + self.min_detectable_effect
        p_avg = (p1 + p2) / 2
        
        n = ((z_alpha * np.sqrt(2 * p_avg * (1 - p_avg)) +
              z_beta  * np.sqrt(p1 * (1 - p1) + p2 * (1 - p2))) ** 2) \
            / (p2 - p1) ** 2
        
        n = int(np.ceil(n))
        return {'per_group': n, 'total': n * 2}


# Define the payment confirmation screen experiment
payment_exp = Experiment(
    name                  = 'payment_confirmation_v2',
    hypothesis            = (
        "IF we show a transaction summary screen before the confirm button, "
        "THEN payment completion rate will increase by ≥5%, "
        "BECAUSE users are more confident when they can review details before submitting."
    ),
    metric                = 'payment_completion_rate',
    metric_type           = 'binary',
    treatment_split       = 0.50,
    min_detectable_effect = 0.05,
    baseline_rate         = 0.68,   # 68% of users currently complete payment
    start_date            = '2026-03-01',
    end_date              = '2026-03-14',
)

sample_size = payment_exp.required_sample_size()

print(f"Experiment: {payment_exp.name}")
print(f"Hypothesis: {payment_exp.hypothesis}")
print(f"\nSample size needed:")
print(f"  Per group: {sample_size['per_group']:,} users")
print(f"  Total:     {sample_size['total']:,} users")
print(f"\n  (Commit to running until this sample size is reached — do NOT peek early)")

## 2. User assignment — verify determinism

In [ ]:
# Verify that the same user always gets the same assignment
test_users = [f'user_{i:04d}' for i in range(10)]

print("Assignment consistency check (same user, 3 separate calls):")
print(f"{'User ID':<12} {'Call 1':<12} {'Call 2':<12} {'Call 3':<12} {'Consistent?'}")
print("-" * 60)

for user in test_users:
    a1 = payment_exp.assign_user(user)
    a2 = payment_exp.assign_user(user)
    a3 = payment_exp.assign_user(user)
    consistent = "✓" if a1 == a2 == a3 else "✗ BUG"
    print(f"{user:<12} {a1:<12} {a2:<12} {a3:<12} {consistent}")

# Check that split is approximately 50/50 across many users
n_check = 10000
assignments = [payment_exp.assign_user(f'user_{i}') for i in range(n_check)]
treatment_pct = assignments.count('treatment') / n_check
print(f"\nSplit across {n_check:,} users: {treatment_pct:.1%} treatment / {1-treatment_pct:.1%} control")

## 3. Simulate experiment data collection

In production, events are logged as users interact with the product.
Here we simulate 14 days of experiment data.

In [ ]:
def simulate_experiment_data(experiment: Experiment, n_users: int = 5000) -> pd.DataFrame:
    """
    Simulate user events during an experiment.
    
    In production, each row in this DataFrame represents one user's
    experience during the experiment window.
    
    Args:
        experiment: The Experiment object defining the test
        n_users:    Number of users to simulate
    
    Returns:
        DataFrame with columns: user_id, group, converted, timestamp
    """
    user_ids   = [f'user_{i:05d}' for i in range(n_users)]
    groups     = [experiment.assign_user(uid) for uid in user_ids]
    
    # Control group: converts at baseline rate (68%)
    # Treatment group: converts at higher rate (simulating a 7% lift)
    true_treatment_lift = 0.07  # The real effect we planted in the simulation
    
    converted = []
    for g in groups:
        if g == 'control':
            converted.append(np.random.binomial(1, experiment.baseline_rate))
        else:
            converted.append(np.random.binomial(1, experiment.baseline_rate + true_treatment_lift))
    
    # Spread events across the experiment window
    start = datetime.strptime(experiment.start_date, '%Y-%m-%d')
    end   = datetime.strptime(experiment.end_date,   '%Y-%m-%d')
    delta_seconds = int((end - start).total_seconds())
    timestamps = [start + timedelta(seconds=np.random.randint(0, delta_seconds))
                  for _ in range(n_users)]
    
    return pd.DataFrame({
        'user_id':   user_ids,
        'group':     groups,
        'converted': converted,
        'timestamp': timestamps
    })

exp_data = simulate_experiment_data(payment_exp, n_users=5000)

summary = exp_data.groupby('group').agg(
    users      =('user_id', 'count'),
    conversions=('converted', 'sum'),
    rate       =('converted', 'mean')
).round(4)

print("Experiment data summary:")
print(summary)
print(f"\nObserved lift: {(summary.loc['treatment','rate'] - summary.loc['control','rate']):.1%}")

## 4. Statistical significance testing

We use a two-proportion z-test for binary outcomes.
The question: is the observed difference real, or could it be random chance?

In [ ]:
def analyse_binary_experiment(data: pd.DataFrame, alpha: float = 0.05) -> dict:
    """
    Run a two-proportion z-test on binary experiment data.
    
    Args:
        data:  DataFrame with 'group' and 'converted' columns
        alpha: Significance threshold (default 0.05 = 5%)
    
    Returns:
        dict with full analysis results
    """
    ctrl = data[data['group'] == 'control']['converted']
    trt  = data[data['group'] == 'treatment']['converted']
    
    # Conversion rates
    p_ctrl = ctrl.mean()
    p_trt  = trt.mean()
    n_ctrl = len(ctrl)
    n_trt  = len(trt)
    
    # Absolute and relative lift
    absolute_lift = p_trt - p_ctrl
    relative_lift = absolute_lift / p_ctrl
    
    # Two-proportion z-test
    # Pooled proportion under H0 (null: both groups have the same rate)
    p_pool = (ctrl.sum() + trt.sum()) / (n_ctrl + n_trt)
    se     = np.sqrt(p_pool * (1 - p_pool) * (1/n_ctrl + 1/n_trt))
    z_stat = absolute_lift / se
    p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))  # Two-tailed
    
    # 95% confidence interval for the absolute lift
    se_diff = np.sqrt(p_ctrl*(1-p_ctrl)/n_ctrl + p_trt*(1-p_trt)/n_trt)
    z_95    = 1.96
    ci_low  = absolute_lift - z_95 * se_diff
    ci_high = absolute_lift + z_95 * se_diff
    
    significant = p_value < alpha
    recommendation = 'SHIP ✓' if (significant and absolute_lift > 0) else 'DO NOT SHIP ✗'
    
    return {
        'control_rate':    round(p_ctrl, 4),
        'treatment_rate':  round(p_trt, 4),
        'absolute_lift':   round(absolute_lift, 4),
        'relative_lift':   round(relative_lift, 4),
        'z_statistic':     round(z_stat, 3),
        'p_value':         round(p_value, 4),
        'ci_95':           (round(ci_low, 4), round(ci_high, 4)),
        'significant':     significant,
        'recommendation':  recommendation,
        'n_control':       n_ctrl,
        'n_treatment':     n_trt,
    }

results = analyse_binary_experiment(exp_data)

print("=" * 55)
print("  EXPERIMENT RESULTS: payment_confirmation_v2")
print("=" * 55)
print(f"  Control rate:    {results['control_rate']:.1%}  (n={results['n_control']:,})")
print(f"  Treatment rate:  {results['treatment_rate']:.1%}  (n={results['n_treatment']:,})")
print(f"  Absolute lift:   {results['absolute_lift']:+.1%}")
print(f"  Relative lift:   {results['relative_lift']:+.1%}")
print(f"  95% CI:          ({results['ci_95'][0]:+.1%}, {results['ci_95'][1]:+.1%})")
print(f"  p-value:         {results['p_value']:.4f}  ({'< 0.05 ✓' if results['p_value'] < 0.05 else '≥ 0.05 ✗'})")
print(f"  Statistically significant: {results['significant']}")
print(f"  → Recommendation: {results['recommendation']}")
print("=" * 55)

## 5. Visualise results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Experiment results: payment_confirmation_v2', fontsize=13)

# 1. Conversion rate comparison with confidence intervals
groups  = ['Control\n(current UI)', 'Treatment\n(summary screen)']
rates   = [results['control_rate'], results['treatment_rate']]
ci_half = (results['ci_95'][1] - results['ci_95'][0]) / 2
errors  = [0.01, ci_half]  # Approximate error bars
colors  = ['#5b8dd9', '#52b788']

bars = axes[0].bar(groups, rates, color=colors, alpha=0.85, width=0.5)
axes[0].errorbar(groups, rates, yerr=errors, fmt='none', color='black', capsize=5, linewidth=2)
for bar, rate in zip(bars, rates):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{rate:.1%}', ha='center', va='bottom', fontweight='bold')
axes[0].set_ylim(0, 1)
axes[0].set_ylabel('Conversion rate')
axes[0].set_title('Conversion rates (95% CI)')

# 2. Confidence interval for lift
ci_low, ci_high = results['ci_95']
axes[1].barh(['Lift'], [results['absolute_lift']], xerr=[[results['absolute_lift']-ci_low],[ci_high-results['absolute_lift']]],
             color='#52b788', alpha=0.85, height=0.3, capsize=8)
axes[1].axvline(0, color='black', linewidth=1, linestyle='--', alpha=0.5)
axes[1].axvline(0.05, color='tomato', linewidth=1, linestyle=':', alpha=0.7, label='MDE (5%)')
axes[1].set_xlabel('Absolute lift')
axes[1].set_title(f'95% CI for lift: ({ci_low:+.1%}, {ci_high:+.1%})')
axes[1].legend()
axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))

# 3. Daily cumulative conversion rate (shows how results stabilised over time)
exp_data_sorted = exp_data.sort_values('timestamp')
for group, color, label in [('control','#5b8dd9','Control'), ('treatment','#52b788','Treatment')]:
    grp = exp_data_sorted[exp_data_sorted['group']==group].copy()
    grp['cumulative_rate'] = grp['converted'].expanding().mean()
    grp = grp.reset_index(drop=True)
    axes[2].plot(grp.index, grp['cumulative_rate'], color=color, label=label, linewidth=2)
axes[2].set_title('Cumulative conversion rate over time')
axes[2].set_xlabel('Users exposed (chronological)')
axes[2].set_ylabel('Running conversion rate')
axes[2].legend()
axes[2].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))

plt.tight_layout()
plt.savefig('experiment_results.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Run a second experiment — a non-significant result

It's just as important to show what a failed experiment looks like.
The framework correctly identifies when a change should NOT ship.

In [ ]:
# Experiment: button colour (small, non-significant effect)
button_exp = Experiment(
    name                  = 'onboarding_cta_green_v1',
    hypothesis            = (
        "IF we change the onboarding CTA button from grey to green, "
        "THEN account activation rate will increase by ≥5%, "
        "BECAUSE green is more prominent and action-oriented."
    ),
    metric                = 'account_activation_rate',
    metric_type           = 'binary',
    baseline_rate         = 0.45,
    min_detectable_effect = 0.05,
    start_date            = '2026-04-01',
    end_date              = '2026-04-14',
)

# Simulate: only a 1.1% true lift — below our MDE of 5%
button_data = simulate_experiment_data.__wrapped__(button_exp, n_users=4000) \
    if hasattr(simulate_experiment_data, '__wrapped__') \
    else pd.DataFrame({
        'user_id':   [f'u_{i}' for i in range(4000)],
        'group':     [button_exp.assign_user(f'u_{i}') for i in range(4000)],
    })

# Manually simulate with small effect
button_data = pd.DataFrame({
    'user_id': [f'u_{i}' for i in range(4000)],
    'group':   [button_exp.assign_user(f'u_{i}') for i in range(4000)],
})
button_data['converted'] = button_data['group'].apply(
    lambda g: np.random.binomial(1, 0.45 + (0.011 if g == 'treatment' else 0))
)

button_results = analyse_binary_experiment(button_data)

print("=" * 55)
print("  EXPERIMENT RESULTS: onboarding_cta_green_v1")
print("=" * 55)
print(f"  Control rate:    {button_results['control_rate']:.1%}")
print(f"  Treatment rate:  {button_results['treatment_rate']:.1%}")
print(f"  Absolute lift:   {button_results['absolute_lift']:+.1%}")
print(f"  p-value:         {button_results['p_value']:.4f}")
print(f"  → Recommendation: {button_results['recommendation']}")
print("=" * 55)
print("\nInterpretation: The small observed lift is likely due to random")
print("variation, not a real effect. We do not ship the green button.")
print("This saves the team from building a false belief into the product.")

## 7. The p-hacking danger — why we don't peek early

This simulation shows what happens if you keep checking p-values and stop as soon
as you see p < 0.05. This is one of the most common mistakes in A/B testing.

In [ ]:
# Simulate: no true effect (both groups have identical rates)
# If we peek at each new user and stop at p < 0.05, how often do we get a false positive?

def simulate_peeking(n_users=2000, n_simulations=100):
    """
    Simulate the p-hacking (optional stopping) problem.
    Shows the false positive rate when you stop as soon as p < 0.05.
    """
    false_positives = 0
    
    for _ in range(n_simulations):
        ctrl = np.random.binomial(1, 0.5, n_users)  # Same rate in both groups
        trt  = np.random.binomial(1, 0.5, n_users)  # No true effect
        
        # Peek at every 50 new users — stop if significant
        for n in range(100, n_users, 50):
            c_sub = ctrl[:n]
            t_sub = trt[:n]
            _, p = stats.ttest_ind(c_sub, t_sub)
            if p < 0.05:
                false_positives += 1
                break  # "Found" significance — stop the experiment
    
    return false_positives / n_simulations

print("Simulating the p-hacking danger (takes ~10 seconds)...")
false_pos_rate = simulate_peeking(n_users=2000, n_simulations=200)

print(f"\nWith optional stopping (peeking every 50 users):")
print(f"  False positive rate: {false_pos_rate:.0%}")
print(f"  Expected rate (no peeking): ~5%")
print(f"  Inflation factor: {false_pos_rate/0.05:.1f}x")
print("\nConclusion: Peeking and stopping early can inflate false positives by 3-5x.")
print("Commit to the pre-calculated sample size before starting.")

## 8. Summary

| Experiment | Metric | Lift | p-value | Decision |
|------------|--------|------|---------|----------|
| Payment confirmation v2 | Completion rate | ~+7% | < 0.05 | **Shipped** |
| CTA button colour | Activation rate | ~+1% | ≥ 0.05 | **Not shipped** |

**Key design principles in this framework:**
- **Deterministic user assignment** — same user always sees the same variant
- **Pre-experiment power calculation** — commit to sample size before starting
- **Two-sided testing** — accounts for both positive and negative effects
- **Confidence intervals, not just p-values** — tells you the range of plausible effects
- **Practical significance check** — a result must clear both statistical and business thresholds to ship